In [1]:
# ============================================================
# Notebook    : 02_sequence_builder.ipynb
# Description : Build Transformer-ready longitudinal sequences
#               - Variable-length sequences with attention mask
#               - Integer-index categorical encoding for embedding layers
#               - Train / validation / test split at IDpol level
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install torch tqdm


# ============================================================
# 1. Common imports
# ============================================================
import pandas as pd
import numpy as np
import torch
import json
import os
import pickle
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

os.makedirs("data/sequences", exist_ok=True)


# ============================================================
# 2. Load preprocessed multi-history data
# ============================================================
df = pd.read_csv("data/fremotor_multi_history_features.csv")

print("Shape:", df.shape)
print(df.head())


# ============================================================
# 3. Build categorical vocabularies
#    - Usage, VehType, VehPower -> integer index for nn.Embedding
#    - Index 0 reserved for padding token (PAD)
#    - Vocabularies saved to json for reproducibility and SHAP
#      interpretation later (index -> category name lookup)
# ============================================================
CAT_COLS = ["Usage", "VehType", "VehPower"]
vocabs = {}

for col in CAT_COLS:
    categories = sorted(df[col].dropna().unique().tolist())
    vocab = {"<PAD>": 0}
    vocab.update({cat: i + 1 for i, cat in enumerate(categories)})
    vocabs[col] = vocab
    df[col + "_idx"] = df[col].map(vocab)

print("\n===== Vocabulary sizes =====")
for col, vocab in vocabs.items():
    print(f"{col:12s} | vocab size (incl. PAD): {len(vocab)}")

with open("data/sequences/vocabs.json", "w", encoding="utf-8") as f:
    json.dump(vocabs, f, ensure_ascii=False, indent=2)
print("\nSaved: vocabs.json")


# ============================================================
# 4. Split IDpol into train / val / test (policyholder-level split)
#    - Splitting by IDpol, not by row, to prevent the same
#      policyholder's sequence from leaking across splits
# ============================================================
np.random.seed(42)

unique_ids = df["IDpol"].unique()
np.random.shuffle(unique_ids)

n = len(unique_ids)
train_ids = unique_ids[: int(n * 0.7)]
val_ids   = unique_ids[int(n * 0.7): int(n * 0.85)]
test_ids  = unique_ids[int(n * 0.85):]

print("\n===== Split sizes (policyholders) =====")
print(f"Train: {len(train_ids):>7,}")
print(f"Val  : {len(val_ids):>7,}")
print(f"Test : {len(test_ids):>7,}")


# ============================================================
# 5. Build variable-length sequence tensors per IDpol
#    - Each policyholder becomes one sequence of shape [T, F]
#    - Pre-grouped once via groupby, then looked up per IDpol
#      (avoids re-scanning the full dataframe 71k+ times)
#    - tqdm progress bar shows build progress
#    - Variable "grouped_data" name deliberately avoids shadowing
#      the built-in dict() function
# ============================================================
NUMERIC_COLS = ["Expo", "YearGap"]
CAT_IDX_COLS = [c + "_idx" for c in CAT_COLS]
LABEL_COL    = "Label"

df_sorted = df.sort_values(["IDpol", "Year"])
grouped_data = {k: v for k, v in df_sorted.groupby("IDpol")}

def build_sequences(ids, grouped_dict, desc="building"):
    sequences = []
    for idpol in tqdm(ids, desc=desc):
        sub = grouped_dict[idpol]
        seq = {
            "IDpol"    : idpol,
            "length"   : len(sub),
            "numeric"  : sub[NUMERIC_COLS].values.astype(np.float32),
            "cat_idx"  : sub[CAT_IDX_COLS].values.astype(np.int64),
            "label"    : sub[LABEL_COL].values.astype(np.int64),
            "year"     : sub["Year"].values.astype(np.int64),
        }
        sequences.append(seq)
    return sequences

train_seqs = build_sequences(train_ids, grouped_data, desc="train")
val_seqs   = build_sequences(val_ids,   grouped_data, desc="val")
test_seqs  = build_sequences(test_ids,  grouped_data, desc="test")

print("\n===== Sequence build check =====")
print(f"Train sequences: {len(train_seqs):>7,}")
print(f"Val   sequences: {len(val_seqs):>7,}")
print(f"Test  sequences: {len(test_seqs):>7,}")
print("\nExample sequence (first train IDpol):")
print("IDpol :", train_seqs[0]["IDpol"])
print("Length:", train_seqs[0]["length"])
print("Numeric shape:", train_seqs[0]["numeric"].shape)
print("Cat idx shape:", train_seqs[0]["cat_idx"].shape)
print("Labels:", train_seqs[0]["label"])


# ============================================================
# 6. Save sequence objects to disk (pickle)
# ============================================================
for name, seqs in [("train", train_seqs), ("val", val_seqs), ("test", test_seqs)]:
    with open(f"data/sequences/{name}_sequences.pkl", "wb") as f:
        pickle.dump(seqs, f)
    print(f"Saved: data/sequences/{name}_sequences.pkl")


# ============================================================
# 7. Collate function for DataLoader
#    - Pads variable-length sequences to the max length in a batch
#    - Returns an attention mask (1 = real token, 0 = padding)
#    - Padding happens only here, keeping the saved pickle files
#      in their original variable-length form
# ============================================================
def collate_fn(batch):
    max_len = max(item["length"] for item in batch)
    batch_size = len(batch)
    n_numeric = batch[0]["numeric"].shape[1]
    n_cat     = batch[0]["cat_idx"].shape[1]

    numeric_padded = torch.zeros(batch_size, max_len, n_numeric, dtype=torch.float32)
    cat_idx_padded = torch.zeros(batch_size, max_len, n_cat, dtype=torch.long)
    label_padded   = torch.zeros(batch_size, max_len, dtype=torch.long)
    attention_mask = torch.zeros(batch_size, max_len, dtype=torch.bool)

    for i, item in enumerate(batch):
        L = item["length"]
        numeric_padded[i, :L] = torch.tensor(item["numeric"])
        cat_idx_padded[i, :L] = torch.tensor(item["cat_idx"])
        label_padded[i, :L]   = torch.tensor(item["label"])
        attention_mask[i, :L] = True

    return {
        "numeric"  : numeric_padded,
        "cat_idx"  : cat_idx_padded,
        "label"    : label_padded,
        "mask"     : attention_mask,
        "IDpol"    : [item["IDpol"] for item in batch],
    }


# ============================================================
# 8. Quick sanity check with a small batch
# ============================================================
class SequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return self.sequences[idx]

train_dataset = SequenceDataset(train_seqs)
train_loader  = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

batch = next(iter(train_loader))
print("\n===== Batch sanity check =====")
print("numeric shape :", batch["numeric"].shape)
print("cat_idx shape :", batch["cat_idx"].shape)
print("label shape   :", batch["label"].shape)
print("mask shape    :", batch["mask"].shape)
print("mask example  :\n", batch["mask"][0])


# ============================================================
# 9. Summary check
# ============================================================
vocab_sizes = {k: len(v) for k, v in vocabs.items()}

print("\n===== Sequence Builder Summary =====")
print(f"Vocab sizes        : {vocab_sizes}")
print(f"Train / Val / Test : {len(train_seqs):,} / {len(val_seqs):,} / {len(test_seqs):,}")
print(f"Numeric features   : {NUMERIC_COLS}")
print(f"Categorical features: {CAT_COLS} (encoded as integer index)")
print("======================================")

Shape: (364997, 11)
      IDpol  Year  NbClaim      Expo    Usage   VehType   VehPower  Label  \
0  PN100006  2003        0  0.669399  Usage15  VehType8  VehPower1      0   
1  PN100006  2004        0  0.997268  Usage15  VehType8  VehPower1      0   
2  PN100006  2005        0  0.997268  Usage15  VehType8  VehPower1      0   
3  PN100006  2006        0  0.997268  Usage15  VehType8  VehPower1      0   
4  PN100006  2007        0  1.000000  Usage15  VehType8  VehPower1      0   

   YearGap  PrevLabel  LabelChanged  
0        0        NaN           NaN  
1        1        0.0           0.0  
2        2        0.0           0.0  
3        3        0.0           0.0  
4        4        0.0           0.0  

===== Vocabulary sizes =====
Usage        | vocab size (incl. PAD): 19
VehType      | vocab size (incl. PAD): 16
VehPower     | vocab size (incl. PAD): 9

Saved: vocabs.json

===== Split sizes (policyholders) =====
Train:  49,974
Val  :  10,709
Test :  10,709


test: 100%|████████████████████████████████████████████████████████████████████| 10709/10709 [00:04<00:00, 2554.25it/s]



===== Sequence build check =====
Train sequences:  49,974
Val   sequences:  10,709
Test  sequences:  10,709

Example sequence (first train IDpol):
IDpol : PN98494
Length: 5
Numeric shape: (5, 2)
Cat idx shape: (5, 3)
Labels: [0 0 0 0 0]
Saved: data/sequences/train_sequences.pkl
Saved: data/sequences/val_sequences.pkl
Saved: data/sequences/test_sequences.pkl

===== Batch sanity check =====
numeric shape : torch.Size([4, 7, 2])
cat_idx shape : torch.Size([4, 7, 3])
label shape   : torch.Size([4, 7])
mask shape    : torch.Size([4, 7])
mask example  :
 tensor([ True,  True,  True,  True, False, False, False])

===== Sequence Builder Summary =====
Vocab sizes        : {'Usage': 19, 'VehType': 16, 'VehPower': 9}
Train / Val / Test : 49,974 / 10,709 / 10,709
Numeric features   : ['Expo', 'YearGap']
Categorical features: ['Usage', 'VehType', 'VehPower'] (encoded as integer index)
